In [174]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta

In [175]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,34276;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=L0cc1T4n3*$*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [176]:
query = """
SELECT DISTINCT v.Pedido,
    v.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2026-02-28'
    AND v.Fecha <= '2026-02-28'
    AND v.Venta_Neta > 0
GROUP BY  v.Pedido,
    v.Cliente,
    c.Nombres + ' ' + c.Apellidos,
    c.Email,
    c.Celular,
    v.Fecha,
    v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query, engine)
df_ventas.head()

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,4354279469,C79785785,DANIEL ABELLO DE CASTRO,dabellodecastro@gmail.com,3223093299,2026-02-28,Retail,258823.53
1,4354279471,C64550066,CARMEN GONZALES,gonzalezcarmen440@gmail.com,3133966429,2026-02-28,Retail,631932.78
2,4354279482,C52249333,MARIA LORENA RIVEROS TELLO,lorena4119@hotmail.com,3144058763,2026-02-28,Retail,607058.83
3,4354279484,C1045682542,KATHLEEN OLANO FORERO,negado@provenzal.net,3044287853,2026-02-28,Retail,199159.66
4,4354279485,C29502197,MARIA EUGENIA MORIBE,mmoribe@gmail.com,3017417439,2026-02-28,Retail,108403.36


In [177]:
# #consulta segmentada 
# query_com = """
# SELECT DISTINCT v.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE  v.fecha >= '2025-10-29'
#     AND v.Fecha <= '2025-10-31'
#     AND v.Venta_Neta > 0
#     AND Descripcion LIKE '%Calendario%'
# GROUP BY  v.Cliente,
#     c.Nombres + ' ' + c.Apellidos,
#     c.Email,
#     c.Celular,
#     v.Fecha,
#     v.Canal
# """
# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_com, engine)
# df_ventas.head()

In [178]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Pedido,
# 	v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta,
#     Canal,
#     SUM(Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE (Codigo_Descuento = '0001 Cumpleaños Cliente 25%' or Codigo_Descuento = 'CUMP_FEB2026')
# 	AND Fecha BETWEEN '2026-02-26' AND '2026-02-28'
# GROUP BY  v.Pedido,
# 	v.Cliente,
#     cc.Nombres + ' ' +  cc.Apellidos,
#     cc.Email,
#     cc.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [179]:
# Leer archivo
df_indigitall = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_indigitall[['phone', 'campaignName', 'sentAt',  'clicks']].copy()

# Renombrar columnas
df_campania.rename(columns={
    'phone': 'Celular',
    'campaignName': 'campania',
    'sentAt': 'fecha_envio',
    'clicks': 'clics'
}, inplace=True)

# Extraer primera fecha de apertura (si viene como lista en string)
df_campania['clics'] = df_campania['clics'].apply(
    lambda x: eval(x)[0] if isinstance(x, str) and x.startswith('[') and len(eval(x)) > 0 else None
)
df_campania['clics'] = pd.to_datetime(df_campania['clics'], errors='coerce').dt.tz_localize(None)
df_campania['fecha_envio'] = pd.to_datetime(df_campania['fecha_envio'], errors='coerce').dt.tz_localize(None).dt.date

df_enviados = df_campania[df_campania['fecha_envio'].notnull()].copy()
df_clics =  df_campania[df_campania['clics'].notnull()].copy()

fecha_maxima = pd.to_datetime('2026-02-28').date()
df_clics_filtrados = df_clics[df_clics['clics'].dt.date <= fecha_maxima]

df_clics_filtrados = df_clics_filtrados.drop_duplicates(subset='Celular', keep='first')

total_clics = df_clics_filtrados['Celular'].nunique()


In [180]:
# df_enviados['Celular'] = df_enviados['Celular'].astype(str)
# df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# # Merge por email
# df_merge = pd.merge(df_ventas, df_enviados, on='Celular', how='inner')

# df_merge

In [181]:
df_enviados['Celular'] = df_enviados['Celular'].astype(str)
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_merge = pd.merge(df_ventas, df_enviados, on='Celular', how='inner')

# Filtrar ventas dentro de 5 días desde la fecha de envio
df_merge['dias_post_apertura'] = (df_merge['Fecha_Venta'] - df_merge['fecha_envio']).apply(lambda x: x.days)
df_atribucion = df_merge[
    (df_merge['dias_post_apertura'] >= 0) &
    (df_merge['dias_post_apertura'] <= 5)
]

# Sumar la venta total atribuida
tottal_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Número de clientes que hicieron clic hasta {fecha_maxima}: {total_clics}")
print(f"Clientes Efectivos: {filas}")
print(f"Total de venta atribuida: {tottal_venta:,.2f}")
# Resultado final
df_atribucion.head()

Número de clientes que hicieron clic hasta 2026-02-28: 2
Clientes Efectivos: 2
Total de venta atribuida: 1,388,025.21


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,campania,fecha_envio,clics,dias_post_apertura
0,4354279780,C71788738,CARLOS MANUEL URIBE MESA,carlosuribe4@yahoo.com,3156658887,2026-02-28,Retail,628571.43,ESTRATEGIA 50& OFF ALMENDRA,2026-02-28,NaT,0
1,4354280066,C42968357,MARGARITA MAZO DE MESA,monicamesamaso07@gmail.com,3508761108,2026-02-28,Retail,759453.78,ESTRATEGIA 50& OFF ALMENDRA,2026-02-28,NaT,0


In [182]:
df_atribucion.to_excel('Ventas.xlsx', index=False)